<div style="background:#1d4e89;color:white;padding:24px 32px;border-radius:8px;margin-bottom:8px">
<h1 style="margin:0;font-size:1.6em">Cyber-Manipulation Susceptibility Pipeline</h1>
<h2 style="margin:6px 0 0;font-weight:normal;font-size:1.1em;opacity:0.85">Custom Ontology Usage Guide — <em>Comprehensive Reference</em></h2>
</div>

This notebook is the **primary reference** for cybersecurity analysts who want to plug **their own PROFILE × ATTACK × OPINION ontologies** into the simulation pipeline.

---

## Contents

| Part | Topic |
|------|-------|
| 1 | Pipeline overview and data model |
| 2 | Ontology format specification |
| 3 | Minimal viable custom ontology |
| 4 | Validation |
| 5 | Programmatic access and leaf inspection |
| 6 | Running the pipeline from CLI |
| 7 | Design guidelines for each ontology type |
| 8 | Advanced: domain-specific worked example |
| 9 | Interpreting outputs |
| 10 | Cost and token estimation |
| 11 | Semantic embedding and UMAP visualization |
| 12 | Scoring new profiles against a fitted artifact |

---

### Prerequisites

1. `OPENROUTER_API_KEY` set in your `.env` file (or environment)
2. Project root in your Python path (handled in the next cell)
3. Dependencies: `pip install -r src/backend/requirements.txt`

In [ ]:
import sys, os, json, textwrap, pprint
from pathlib import Path

# ── Add project root to path ──────────────────────────────────────────────
PROJECT_ROOT = Path("../../../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Load environment variables ────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

has_api_key = bool(os.getenv("OPENROUTER_API_KEY"))
print(f"Project root  : {PROJECT_ROOT}")
print(f"API key loaded: {has_api_key}")
if not has_api_key:
    print("  → Set OPENROUTER_API_KEY in your .env file to run live simulations.")
    print("  → Validation and dry-run cells work without an API key.")

---

## Part 1 — Pipeline overview and data model

The pipeline simulates **inter-individual differences in susceptibility to cyber-manipulation**. It takes three hierarchical ontologies as input and produces a research-grade statistical analysis as output.

### The three ontologies

| Ontology | Role | Encodes |
|----------|------|---------|
| **PROFILE** | Target population | Psychological traits, demographics, digital literacy, political engagement |
| **ATTACK** | Threat taxonomy | Adversarial content types and mechanisms |
| **OPINION** | State space | Political/security beliefs being targeted, with adversarial direction per leaf |

### What the pipeline computes

For each `(profile, attack, opinion)` triple the pipeline:

1. **Generates a pseudoprofile** — a synthetic individual whose psychology, demographics, and beliefs are consistent with the profile ontology leaves
2. **Elicits a baseline opinion** — on a [-1000, 1000] scale, before any attack exposure
3. **Generates an attack message** — platform-native manipulative content targeting that opinion leaf
4. **Reviews realism** — self-supervised LLM audit, repair loop if below threshold
5. **Elicits post-exposure opinion** — same agent, same leaf, after reading the attack
6. **Reviews coherence** — plausibility and boundedness checks
7. **Computes adversarial effectivity** — `AE = signed_delta × adversarial_direction`; positive = attack succeeded

Statistical analysis then estimates which profile dimensions **moderate susceptibility** across the full factorial design.

### Primary outcome

```
adversarial_effectivity_ik = (post_score_ik − baseline_score_ik) × direction_k

direction_k ∈ {+1, −1}   (from OPINION ontology; 0 = excluded)
```

Positive AE = opinion shifted in the adversary's intended direction.

---

## Part 2 — Ontology format specification

Each ontology is a **hierarchical JSON object**. The same rules apply to all three:

### Node type rules

| Node type | How identified | Example |
|-----------|---------------|--------|
| **Subtree** | Key starts with uppercase | `"Big_Five": { ... }` |
| **Leaf** | Empty dict `{}` OR dict with only metadata keys | `"neuroticism_pct": {}` |
| **Metadata key** | Lowercase, starts with `_`, or in known set | `"adversarial_direction": -1` |

### Reserved metadata keys

```json
{
  "adversarial_direction":  1,      // OPINION only: +1 / -1 / 0
  "description":            "...",  // Human-readable description (used in LLM prompts)
  "mechanism":              "...",  // ATTACK only: psychological mechanism
  "platform_hint":          "...",  // ATTACK only: e.g. "social media post"
  "notes":                  "...",  // Free-form internal notes
  "examples":               "..."   // Concrete examples (passed to LLM)
}
```

### PROFILE leaf naming conventions

The pipeline infers column types from leaf names:

| Pattern | Inferred type | Encoding |
|---------|--------------|----------|
| Ends with `_pct` | Continuous (0–100 percentile) | Float |
| Contains `age`, `years`, `income` | Absolute numeric | Float |
| Children are CamelCase level names | Categorical | One-hot |

Example:
```json
"neuroticism_pct": {},         // → float column
"age_years": {},               // → float column (absolute scale)
"education_level": {           // → one-hot columns
    "High_School": {},
    "Undergraduate": {},
    "Graduate": {}
}
```

In [ ]:
# Inspect the test ontologies shipped with the project
test_ont_root = PROJECT_ROOT / "src" / "backend" / "ontology" / "separate" / "test"

with open(test_ont_root / "ATTACK"  / "attack.json")  as f: attack_tree  = json.load(f)
with open(test_ont_root / "OPINION" / "opinion.json") as f: opinion_tree = json.load(f)
with open(test_ont_root / "PROFILE" / "profile.json") as f: profile_tree = json.load(f)

from src.backend.utils.ontology_utils import flatten_leaf_paths
attack_leaves  = flatten_leaf_paths(attack_tree)
opinion_leaves = flatten_leaf_paths(opinion_tree)
profile_leaves = flatten_leaf_paths(profile_tree)

print(f"Test ontology dimensions")
print(f"  ATTACK  leaves : {len(attack_leaves):3d}")
print(f"  OPINION leaves : {len(opinion_leaves):3d}")
print(f"  PROFILE leaves : {len(profile_leaves):3d}")
print(f"  Factorial tasks: {len(attack_leaves) * len(opinion_leaves):3d} (attack × opinion)")
print()
print("Sample ATTACK leaves:")
for leaf in attack_leaves[:6]: print(f"  {leaf}")
print()
print("Sample OPINION leaves:")
for leaf in opinion_leaves[:6]: print(f"  {leaf}")

---

## Part 3 — Minimal viable custom ontology

The smallest valid triplet requires:
- **PROFILE**: ≥ 5 leaves, depth ≥ 2, at least one continuous and one categorical column
- **ATTACK**: ≥ 1 leaf (≥ 2 recommended for comparative analysis)
- **OPINION**: ≥ 1 leaf with `adversarial_direction` ≠ 0 (≥ 4 recommended)

The example below models a **health misinformation** domain — a minimal but fully functional triplet.

In [ ]:
import tempfile

# ── Minimal PROFILE ───────────────────────────────────────────────────────
minimal_profile = {
    "Individual_Characteristics": {
        "Psychological_Traits": {
            "Trait_Anxiety": {
                "anxiety_score_pct": {},        # continuous: 0-100 percentile
                "worry_frequency_pct": {}
            },
            "Need_for_Cognition": {
                "need_for_cognition_pct": {},    # higher = more analytical
                "analytical_preference_pct": {}
            }
        },
        "Demographics": {
            "age_years": {},                    # absolute numeric
            "education_level": {                # categorical
                "No_Formal":      {},
                "High_School":    {},
                "Undergraduate":  {},
                "Graduate":       {}
            }
        }
    }
}

# ── Minimal ATTACK ────────────────────────────────────────────────────────
minimal_attack = {
    "ATTACK_VECTORS": {
        "Health_Disinformation": {
            "False_Statistic_Injection": {
                "mechanism": "Fabricated statistics attributed to credible medical sources",
                "description": "Insert plausible-sounding but false numerical claims into social media posts",
                "platform_hint": "Twitter/X post with cited fake study"
            },
            "Fear_Appeal_Health_Post": {
                "mechanism": "Threat framing with emotional amplification targeting health anxieties",
                "description": "Exaggerate negative health consequences to trigger fear and urgency around medical decisions",
                "platform_hint": "Facebook post with alarming imagery"
            }
        }
    }
}

# ── Minimal OPINION ───────────────────────────────────────────────────────
minimal_opinion = {
    "Public_Health": {
        "Vaccine_Policy": {
            "Mandatory_Vaccination_Support": {
                "adversarial_direction": -1,    # adversary wants to REDUCE vaccine support
                "description": "Support for government-mandated vaccination programmes"
            },
            "Healthcare_Worker_Exemption_Support": {
                "adversarial_direction": 1,     # adversary wants to INCREASE exemption support
                "description": "Support for allowing healthcare workers to opt out of vaccination"
            }
        },
        "Health_Authority_Trust": {
            "WHO_Institutional_Trust": {
                "adversarial_direction": -1,
                "description": "Trust in the World Health Organization"
            },
            "Alternative_Medicine_Legitimacy": {
                "adversarial_direction": 1,
                "description": "Perceived legitimacy of alternative medicine as a substitute for evidence-based care"
            }
        }
    }
}

# Save to a temp directory
tmp_dir = Path(tempfile.mkdtemp(prefix="cog_minimal_"))
for subdir, obj, fname in [
    ("PROFILE", minimal_profile, "profile.json"),
    ("ATTACK",  minimal_attack,  "attack.json"),
    ("OPINION", minimal_opinion, "opinion.json"),
]:
    d = tmp_dir / subdir
    d.mkdir()
    with open(d / fname, "w") as fh:
        json.dump(obj, fh, indent=2)

print(f"Minimal ontologies written to: {tmp_dir}")
print(f"  {len(flatten_leaf_paths(minimal_attack))} attack leaves")
print(f"  {len(flatten_leaf_paths(minimal_opinion))} opinion leaves  (all with adversarial_direction)")
print(f"  {len(flatten_leaf_paths(minimal_profile))} profile leaves")

---

## Part 4 — Validating your ontologies

Always validate before running a simulation. The validator catches:
- Structural errors (missing leaves, wrong types, invalid `adversarial_direction` values)
- Depth warnings (leaves too shallow for meaningful hierarchical decomposition)
- Cross-ontology sanity checks (total task count, minimum profile dimensionality)
- Metadata completeness (missing `mechanism`/`description` in ATTACK leaves)

**Return codes:** `is_valid=True` means zero errors (warnings are non-blocking).

In [ ]:
from src.backend.user_ontology.validator import validate_ontology_triplet

report = validate_ontology_triplet(
    profile_path=tmp_dir / "PROFILE" / "profile.json",
    attack_path=tmp_dir  / "ATTACK"  / "attack.json",
    opinion_path=tmp_dir / "OPINION" / "opinion.json",
)

print(report.summary())
print()
if report.is_valid:
    print("[PASS] Ontologies are valid — safe to proceed.")
    if report.warnings:
        print(f"  {len(report.warnings)} warning(s) above — review before production runs.")
else:
    print("[FAIL] Fix the errors above before running the pipeline.")

In [ ]:
# ── Demonstrate error catching ────────────────────────────────────────────
# An OPINION leaf with an invalid adversarial_direction should raise an error
bad_opinion = {
    "Domain": {
        "Topic": {
            "My_Opinion_Leaf": {"adversarial_direction": 99}  # invalid!
        }
    }
}
bad_opinion_path = tmp_dir / "OPINION" / "bad_opinion.json"
bad_opinion_path.write_text(json.dumps(bad_opinion, indent=2))

bad_report = validate_ontology_triplet(
    profile_path=tmp_dir / "PROFILE" / "profile.json",
    attack_path=tmp_dir  / "ATTACK"  / "attack.json",
    opinion_path=bad_opinion_path,
)
print(bad_report.summary())

---

## Part 5 — Programmatic access and leaf inspection

Before running a simulation, inspect the resolved leaves and adversarial directions.

In [ ]:
from src.backend.user_ontology import load_user_ontology_triplet
from src.backend.utils.ontology_utils import flatten_leaf_paths, load_adversarial_directions_from_opinion

triplet = load_user_ontology_triplet(
    profile_path=tmp_dir / "PROFILE" / "profile.json",
    attack_path=tmp_dir  / "ATTACK"  / "attack.json",
    opinion_path=tmp_dir / "OPINION" / "opinion.json",
    validate=True,
)

attack_leaves  = flatten_leaf_paths(triplet["ATTACK"])
opinion_leaves = flatten_leaf_paths(triplet["OPINION"])
profile_leaves = flatten_leaf_paths(triplet["PROFILE"])
adv_dirs, _   = load_adversarial_directions_from_opinion(triplet["OPINION"])

print("Attack leaves:")
for leaf in attack_leaves:
    label = leaf.split(">")[-1].strip()
    print(f"  {label}")

print("\nOpinion leaves and adversarial directions:")
for leaf in opinion_leaves:
    label = leaf.split(">")[-1].strip()
    direction = adv_dirs.get(label, "unknown")
    direction_str = {1: "→ adversary wants INCREASE", -1: "→ adversary wants DECREASE", 0: "→ excluded from AE"}.get(direction, "?")
    print(f"  {label:<45}  dir={direction:+d}  {direction_str}")

print(f"\nTotal attack × opinion tasks: {len(attack_leaves) * len(opinion_leaves)}")
print(f"Profile feature dimensions  : {len(profile_leaves)}")

---

## Part 6 — Running the pipeline from CLI

The recommended way to run a custom simulation. The CLI:
1. Validates your ontologies
2. Auto-discovers all attack leaves (or uses `--attack-leaves` to select a subset)
3. Builds a temporary ontology root with the standard directory structure
4. Delegates to `run_full_pipeline.py` with the correct parameters

### Dry-run first (no API calls, no simulation)

In [ ]:
import subprocess

dry_run_cmd = [
    sys.executable, "-m", "src.backend.user_ontology.cli",
    "--profile-json", str(tmp_dir / "PROFILE" / "profile.json"),
    "--attack-json",  str(tmp_dir / "ATTACK"  / "attack.json"),
    "--opinion-json", str(tmp_dir / "OPINION" / "opinion.json"),
    "--run-id",         "notebook_health_demo",
    "--output-root",    "evaluation/notebook_health_demo",
    "--n-profiles",     "20",
    "--openrouter-model", "mistralai/mistral-small-3.2-24b-instruct",
    "--bootstrap-samples", "200",
    "--dry-run",
]

result = subprocess.run(dry_run_cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# ── Validate-only mode (useful in CI / pre-flight checks) ─────────────────
validate_cmd = [
    sys.executable, "-m", "src.backend.user_ontology.cli",
    "--profile-json", str(tmp_dir / "PROFILE" / "profile.json"),
    "--attack-json",  str(tmp_dir / "ATTACK"  / "attack.json"),
    "--opinion-json", str(tmp_dir / "OPINION" / "opinion.json"),
    "--validate-only",
]

result = subprocess.run(validate_cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
print(result.stdout)
print(f"Return code: {result.returncode}  (0 = valid, 1 = errors)")

### Run a live simulation

> **Note:** The cell below makes real API calls and incurs costs. See Part 10 for a cost estimate before running. Requires `OPENROUTER_API_KEY` in your environment.

```python
# Uncomment and run after reviewing the dry-run output above

live_cmd = [
    sys.executable, "-m", "src.backend.user_ontology.cli",
    "--profile-json", str(tmp_dir / "PROFILE" / "profile.json"),
    "--attack-json",  str(tmp_dir / "ATTACK"  / "attack.json"),
    "--opinion-json", str(tmp_dir / "OPINION" / "opinion.json"),
    "--run-id",         "health_demo",
    "--output-root",    "evaluation/health_demo",
    "--n-profiles",     "40",
    "--openrouter-model", "mistralai/mistral-small-3.2-24b-instruct",
    "--bootstrap-samples", "200",
    "--max-concurrency",   "10",
]

result = subprocess.run(live_cmd, cwd=str(PROJECT_ROOT))
print("Exit code:", result.returncode)
```

---

## Part 7 — Design guidelines

### PROFILE design

| Guideline | Why |
|-----------|-----|
| Include ≥ 2 psychological inventory subtrees | Enables hierarchical feature decomposition across different trait dimensions |
| Mix `_pct` continuous leaves with categorical leaves | Required for typed column inference; models both trait distributions and discrete demographics |
| Use `age_years` if age-related effects matter | Absolute numeric encoding; detected by token `age`/`years` |
| Include facet-level leaves (e.g. `anxiety_pct`, `worry_pct` under `Trait_Anxiety`) | Finer-grained variance decomposition in the conditional susceptibility index |
| Target ≥ 15–20 feature dimensions | Ridge regression reliability degrades sharply below this threshold |

### ATTACK design

| Guideline | Why |
|-----------|-----|
| Add `mechanism` and `description` to every leaf | Passed directly to the attack generation LLM prompt; strongly influences realism scores |
| Add `platform_hint` when platform matters | E.g. "WhatsApp forward" vs. "Twitter/X post" — shapes tone and format |
| Group related attacks under shared subtrees | Enables within-family vs. cross-family attack comparison |
| Include ≥ 2 mechanism families | Allows cross-family moderation analysis |

### OPINION design

| Guideline | Why |
|-----------|-----|
| Set `adversarial_direction: ±1` on every target leaf | Without direction, AE falls back to `\|Δ\|` which loses sign information |
| Use `adversarial_direction: 0` for neutral/ambiguous leaves | Included in baseline state space but excluded from directional scoring |
| Include ≥ 2 opinion domains | Cross-domain moderation comparison reveals domain-general vs. domain-specific effects |
| Use substantive leaf names | E.g. `NATO_Collective_Defense_Support` not `Opinion_Leaf_1` — names appear in figures |
| Add `description` to each leaf | Shapes the opinion elicitation prompt; improves construct validity |

---

## Part 8 — Advanced: domain-specific worked example

This section builds a **financial fraud susceptibility** ontology — a richer example using all recommended metadata fields and covering multiple attack families and opinion domains.

In [ ]:
financial_profile = {
    "Financial_Vulnerability": {
        "Cognitive_Biases": {
            "Loss_Aversion": {
                "loss_aversion_pct": {},         # continuous
                "risk_sensitivity_pct": {}
            },
            "Overconfidence": {
                "financial_overconfidence_pct": {},
                "market_prediction_confidence_pct": {}
            }
        },
        "Financial_Literacy": {
            "Basic_Financial_Knowledge": {
                "compound_interest_knowledge_pct": {},
                "investment_risk_understanding_pct": {}
            },
            "Scam_Detection_Ability": {
                "phishing_detection_pct": {},
                "impersonation_awareness_pct": {}
            }
        },
        "Demographics": {
            "age_years": {},
            "income_usd": {},                    # absolute numeric
            "education_level": {
                "No_Formal": {},
                "High_School": {},
                "Undergraduate": {},
                "Graduate": {}
            }
        }
    }
}

financial_attack = {
    "ATTACK_VECTORS": {
        "Investment_Fraud": {
            "Pump_and_Dump_Promotion": {
                "mechanism": "Artificial scarcity and social proof around micro-cap assets",
                "description": "Posts promoting a little-known stock/crypto with fabricated insider information",
                "platform_hint": "Telegram investment group or Twitter/X finance thread"
            },
            "Guaranteed_Return_Offer": {
                "mechanism": "Exploitation of financial anxiety with too-good-to-be-true returns",
                "description": "Promises risk-free returns significantly above market rate",
                "platform_hint": "Sponsored social media ad or email newsletter"
            }
        },
        "Social_Engineering_Finance": {
            "Authority_Impersonation": {
                "mechanism": "Fake institutional authority to bypass skepticism",
                "description": "Impersonates a bank or financial regulator to request urgent action",
                "platform_hint": "Email or SMS from spoofed institutional sender"
            },
            "Romance_Pig_Butchering": {
                "mechanism": "Relationship trust exploitation leading to crypto investment",
                "description": "Builds romantic rapport over weeks before introducing a fraudulent investment platform",
                "platform_hint": "Dating app or social media DM"
            }
        }
    }
}

financial_opinion = {
    "Financial_Security": {
        "Investment_Behavior": {
            "Risky_Asset_Allocation_Support": {
                "adversarial_direction": 1,
                "description": "Willingness to allocate a large share of savings to high-risk assets"
            },
            "Diversification_Discipline": {
                "adversarial_direction": -1,
                "description": "Commitment to diversified portfolio strategy rather than concentrated bets"
            }
        },
        "Institution_Trust": {
            "Regulatory_Authority_Trust": {
                "adversarial_direction": -1,
                "description": "Trust in financial regulators to protect investors"
            },
            "Alternative_Platform_Appeal": {
                "adversarial_direction": 1,
                "description": "Appeal of unregulated alternative investment platforms"
            }
        }
    }
}

# Save and validate
fin_dir = Path(tempfile.mkdtemp(prefix="cog_finance_"))
for subdir, obj, fname in [
    ("PROFILE", financial_profile, "profile.json"),
    ("ATTACK",  financial_attack,  "attack.json"),
    ("OPINION", financial_opinion, "opinion.json"),
]:
    d = fin_dir / subdir; d.mkdir()
    (d / fname).write_text(json.dumps(obj, indent=2))

fin_report = validate_ontology_triplet(
    fin_dir / "PROFILE" / "profile.json",
    fin_dir / "ATTACK"  / "attack.json",
    fin_dir / "OPINION" / "opinion.json",
)
print(fin_report.summary())

fin_triplet = load_user_ontology_triplet(
    fin_dir / "PROFILE" / "profile.json",
    fin_dir / "ATTACK"  / "attack.json",
    fin_dir / "OPINION" / "opinion.json",
    validate=False,
)
n_attack  = len(flatten_leaf_paths(fin_triplet["ATTACK"]))
n_opinion = len(flatten_leaf_paths(fin_triplet["OPINION"]))
n_profile = len(flatten_leaf_paths(fin_triplet["PROFILE"]))
print(f"\nFinancial fraud ontology: {n_attack} attack × {n_opinion} opinion = {n_attack*n_opinion} tasks, {n_profile} profile dims")

---

## Part 9 — Interpreting outputs

After a successful run, outputs are in `evaluation/<run_id>/`:

| Path | Content |
|------|---------|
| `stage_outputs/05_compute_effectivity_deltas/sem_long_encoded.csv` | Full panel: one row per (profile, attack, opinion), with adversarial effectivity |
| `stage_outputs/06_construct_structural_equation_model/` | SEM results, conditional susceptibility artifact, CSI scores |
| `stage_outputs/07_generate_research_visuals/interactive_sem_dashboard.html` | Interactive Plotly dashboard |
| `paper/publication_assets/figures/` | Static figures for manuscript |

### Key metrics

| Metric | Interpretation |
|--------|----------------|
| `adversarial_effectivity` | Primary outcome. Positive = attack succeeded. Range: [-2000, +2000] |
| `abs_delta_score` | Raw opinion shift magnitude. Always positive. Direction-blind. |
| `susceptibility_index_pct` | Profile-level aggregate susceptibility, 0–100 percentile rank |
| `marginal_r2_*` | Unique variance explained by each feature group in the CSI ridge models |
| `icc1_*` | Between-profile variance fraction for each outcome (ICC > 0.05 = meaningful profile effect) |

### Interpreting SEM results

The SEM estimates **paths from each profile moderator to each adversarial effectivity indicator**. A positive path coefficient for a trait (e.g. Extraversion → AE) means profiles scoring higher on that trait tend to shift more in the adversary's intended direction.

In [ ]:
# ── Load and inspect panel from a completed run ───────────────────────────
import pandas as pd

run_id = "run_9"  # change to your run
panel_path = PROJECT_ROOT / "evaluation" / run_id / "stage_outputs" / "05_compute_effectivity_deltas" / "sem_long_encoded.csv"

if panel_path.exists():
    df = pd.read_csv(panel_path)
    print(f"Panel shape: {df.shape}")
    print(f"\nColumns: {list(df.columns[:10])}  ... (truncated)")
    print(f"\nAdversarial effectivity summary:")
    print(df["adversarial_effectivity"].describe().round(2))
    print(f"\nPositive AE rate: {(df['adversarial_effectivity'] > 0).mean():.1%}")
    print(f"Attack breakdown:")
    print(df.groupby("attack_leaf")["adversarial_effectivity"].mean().sort_values().round(2))
else:
    print(f"No run found at {panel_path}")
    print("  → Run the pipeline first, then re-execute this cell.")

In [ ]:
# ── Load CSI scores ───────────────────────────────────────────────────────
csi_path = PROJECT_ROOT / "evaluation" / run_id / "stage_outputs" / "06_construct_structural_equation_model" / "conditional_susceptibility_artifact.json"

if csi_path.exists():
    with open(csi_path) as f:
        artifact = json.load(f)
    print("CSI artifact keys:", list(artifact.keys())[:8])
    print()
    if "profile_scores" in artifact:
        scores = pd.DataFrame(artifact["profile_scores"])
        print("Top 5 most susceptible profiles:")
        print(scores.nlargest(5, "csi_pct")[["profile_id", "csi_pct"]].to_string(index=False))
    if "hierarchical_decomposition" in artifact:
        decomp = artifact["hierarchical_decomposition"]
        print("\nHierarchical R² decomposition (mean across tasks):")
        for group, val in sorted(decomp.items(), key=lambda x: -x[1]):
            print(f"  {group:<35} {val:.4f}")
else:
    print(f"No artifact at {csi_path}")
    print("  → Run the pipeline through stage 06 first.")

---

## Part 10 — Cost and token estimation

Each simulated scenario makes **5 LLM calls**:

| Stage | Call | Approx. tokens |
|-------|------|---------------|
| Stage 01 | Profile generation | ~800 |
| Stage 02 | Baseline opinion elicitation | ~400 |
| Stage 03 | Attack message generation + realism review | ~700 |
| Stage 04 | Post-exposure opinion elicitation | ~500 |
| Review | Coherence check | ~300 |

**Total per scenario: ~2,700 tokens**

In [ ]:
def estimate_cost(
    n_profiles: int,
    n_attacks: int,
    n_opinions: int,
    tokens_per_scenario: int = 2700,
    price_per_million: float = 0.15,  # Mistral Small 3.2 via OpenRouter (approx)
) -> None:
    n_scenarios = n_profiles * n_attacks * n_opinions
    total_tokens = n_scenarios * tokens_per_scenario
    cost_usd = total_tokens / 1_000_000 * price_per_million
    print(f"Design: {n_profiles} profiles × {n_attacks} attacks × {n_opinions} opinions")
    print(f"  Scenarios    : {n_scenarios:,}")
    print(f"  Tokens (est) : {total_tokens:,.0f}  ({total_tokens/1e6:.1f}M)")
    print(f"  Cost (est)   : ${cost_usd:.2f}  at ${price_per_million}/M tokens")
    print()

print("=== Cost estimates ===")
print()
estimate_cost(n_profiles=20,  n_attacks=2, n_opinions=4,  price_per_million=0.15)  # minimal health demo
estimate_cost(n_profiles=40,  n_attacks=4, n_opinions=4,  price_per_million=0.15)  # small study
estimate_cost(n_profiles=80,  n_attacks=6, n_opinions=8,  price_per_million=0.15)  # current study design
estimate_cost(n_profiles=200, n_attacks=6, n_opinions=10, price_per_million=0.15)  # scale-up
print("Note: Prices are approximate. Check OpenRouter pricing for your model.")
print("Tip : Start with n_profiles=20 to validate output quality before scaling up.")

---

## Part 11 — Semantic embedding and UMAP visualization

The pipeline can embed all ontology leaves using `text-embedding-3-large` (OpenRouter) and project them to 2D via UMAP. This reveals semantic similarity structure — e.g. which attack mechanisms cluster together, which opinion leaves are semantically related.

The resulting `embedding_dashboard.json` is loaded automatically by the interactive dashboard in the **Ontologies → Semantic Embedding Space** tab.

> **Requires:** `OPENROUTER_API_KEY` and the embedding model accessible on your account.

In [ ]:
# ── Semantic embedding (live — requires API key) ──────────────────────────
# Uncomment the block below to run

# from src.backend.utils.semantic_embedding import embed_ontology
#
# artifact = embed_ontology(
#     ontology_root=str(test_ont_root),
#     out_dir="evaluation/embedding_demo",
#     n_clusters=6,
# )
# artifact.write("evaluation/embedding_demo")
# print(f"Embedding complete. {len(artifact.leaves)} leaves embedded.")
# print(f"Cluster assignments: {artifact.cluster_labels[:10]}")

# ── Preview embedding utility interface ──────────────────────────────────
import inspect
try:
    from src.backend.utils.semantic_embedding import embed_ontology
    sig = inspect.signature(embed_ontology)
    print("embed_ontology signature:")
    for name, param in sig.parameters.items():
        default = f" = {param.default!r}" if param.default is not inspect.Parameter.empty else ""
        print(f"  {name}{default}")
except ImportError as e:
    print(f"semantic_embedding not available: {e}")
    print("  → Install umap-learn and sentence-transformers: pip install umap-learn sentence-transformers")

---

## Part 12 — Scoring new profiles against a fitted artifact

After a completed run, the conditional susceptibility artifact can be used to **score new profiles without re-running the full simulation**. This is useful for:
- Comparing hypothetical profiles (e.g. "how susceptible would a high-Agreeableness vs. low-Agreeableness individual be?")
- Operational threat assessment: given a target profile, predict susceptibility under specific attack × opinion configurations
- Ablation studies: vary one feature while holding all others constant

In [ ]:
# ── Score new profiles from an existing artifact ──────────────────────────
from src.backend.utils.conditional_susceptibility import (
    fit_conditional_susceptibility_index,
    score_profiles_with_conditional_artifact,
)

run_id = "run_9"
panel_path = PROJECT_ROOT / "evaluation" / run_id / "stage_outputs" / "05_compute_effectivity_deltas" / "sem_long_encoded.csv"

if panel_path.exists():
    df = pd.read_csv(panel_path)
    
    # Re-fit (or load) the artifact
    fit_result = fit_conditional_susceptibility_index(
        df,
        outcome_metric="adversarial_effectivity",
        seed=42,
        compute_hierarchy=True,
    )
    artifact = fit_result.artifact
    
    # Score the existing profiles
    profile_features = df[["profile_id"] + artifact.feature_columns].drop_duplicates()
    scores, breakdown = score_profiles_with_conditional_artifact(profile_features, artifact)
    
    print(f"Scored {len(scores)} profiles against {len(artifact.task_models)} tasks")
    print(f"\nCSI distribution:")
    print(scores["csi_pct"].describe().round(2))
    print(f"\nTop 3 most susceptible:")
    print(scores.nlargest(3, "csi_pct")[["profile_id", "csi_pct"]].to_string(index=False))
    print(f"\nBottom 3 most resistant:")
    print(scores.nsmallest(3, "csi_pct")[["profile_id", "csi_pct"]].to_string(index=False))
else:
    print(f"No panel found at {panel_path}")
    print("  → Run the pipeline through stage 05 first.")

In [ ]:
# ── Score a hypothetical new profile via the CLI ──────────────────────────
# This calls score_profile.py with individual feature flags.
# Available flags depend on the feature columns in your artifact.

artifact_path = (
    PROJECT_ROOT / "evaluation" / "run_9" /
    "stage_outputs" / "06_construct_structural_equation_model" /
    "conditional_susceptibility_artifact.json"
)

if artifact_path.exists():
    score_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "backend" / "pipeline" / "separate" /
            "compute_conditional_susceptibility" / "score_profile.py"),
        "--artifact-path", str(artifact_path),
        "--age", "34",
        "--sex", "Male",
        "--neuroticism-pct", "75",
        "--conscientiousness-pct", "20",
        "--extraversion-pct", "85",
        "--agreeableness-pct", "60",
        "--openness-pct", "70",
    ]
    result = subprocess.run(score_cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    print(result.stdout[:2000])  # first 2000 chars
    if result.returncode != 0:
        print("STDERR:", result.stderr[:500])
else:
    print(f"Artifact not found at {artifact_path}")
    print("  → Run through stage 06 first.")

---

## Summary

| Step | Command / API |
|------|---------------|
| Validate | `validate_ontology_triplet(profile, attack, opinion)` |
| Inspect leaves | `flatten_leaf_paths(tree)` + `load_adversarial_directions_from_opinion(tree)` |
| Dry run | `python -m src.backend.user_ontology.cli ... --dry-run` |
| Validate only | `python -m src.backend.user_ontology.cli ... --validate-only` |
| Live run | `python -m src.backend.user_ontology.cli ... --n-profiles 40` |
| Score new profile | `score_profiles_with_conditional_artifact(features, artifact)` |
| Embed leaves | `embed_ontology(ontology_root, out_dir, n_clusters)` |

---

<div style="background:#f0f4f8;padding:16px 20px;border-radius:6px;border-left:4px solid #1d4e89">
<b>Questions and feedback:</b> Open an issue at <a href="https://github.com/stvsever/research_paper_on_cognitive_sovereignity">github.com/stvsever/research_paper_on_cognitive_sovereignity</a><br>
<b>Paper:</b> Van Severen, S., De Schryver, T., &amp; Ostyn, M. (2025). <em>Inter-individual Differences in Susceptibility to Cyber-manipulation</em>. Ghent University.
</div>